# Horizon experiment: final figures

Forecast horizons of 1, 5, 10, 15 and 20 weekdays. h=20 is the official M6
horizon; the others are extensions.

All values are computed by `Horizon Experiment/horizon_diagnostics.py` and read
from `Horizon Experiment/results/`. This notebook only draws the figures.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd()
ROOT = HERE.parent
# computation and results stay in Horizon Experiment/; only the figures live here
RESULTS = ROOT / "Horizon Experiment" / "results"
FIGURES = HERE / "figures"
TABLES = HERE / "tables"
FIGURES.mkdir(exist_ok=True)
TABLES.mkdir(exist_ok=True)

HORIZONS = [1, 5, 10, 15, 20]
NAIVE_RPS = 0.16
EXCLUDED = (4, 1)   # h=1 uses 11 rounds: Round 4 is excluded

COLOURS = {"Chronos-2 120M (independent sampling)": "#1f77b4",
           "Financial Chronos 46M": "#2ca02c",
           "Chronos-T5 Base 200M": "#d62728"}
SHORT = {"Chronos-2 120M (independent sampling)": "Chronos-2",
         "Financial Chronos 46M": "Financial Chronos",
         "Chronos-T5 Base 200M": "Chronos-T5 Base"}

summary = pd.read_csv(RESULTS / "horizon_rps_summary.csv")
detail = pd.read_csv(RESULTS / "horizon_forecast_detail.csv")
diagnostics = pd.read_csv(RESULTS / "horizon_diagnostics.csv")
ECE_COLUMN = next(c for c in diagnostics.columns if c.startswith("ece_"))

confidence = detail[~((detail["round"] == EXCLUDED[0])
                      & (detail["horizon_days"] == EXCLUDED[1]))]

print(f"{len(detail)} detail rows, horizons {sorted(detail['horizon_days'].unique())}")

## Horizon Experiment Results

h=1 uses 11 rounds (Round 4 excluded). Accuracy is 5-class argmax accuracy,
chance 0.2. ECE uses 5 uniform bins.

In [ ]:
table = diagnostics.assign(model=diagnostics["model"].map(SHORT)).rename(columns={
    "horizon_days": "horizon",
    "overall_mean_rps_excl_degenerate": "RPS",
    "mean_confidence": "confidence",
    "mean_entropy": "entropy",
    ECE_COLUMN: "ECE",
})[["model", "horizon", "RPS", "confidence", "entropy", "accuracy", "ECE"]]
table = table.sort_values(["model", "horizon"]).round(4).reset_index(drop=True)
table

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.9))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.axis("off")

rendered = ax.table(cellText=[[f"{v:.4f}" if isinstance(v, float) else str(v)
                               for v in row] for row in table.to_numpy()],
                    colLabels=table.columns, cellLoc="center", loc="center")
rendered.auto_set_font_size(False)
rendered.set_fontsize(9)
rendered.scale(1, 1.45)

for (row, column), cell in rendered.get_celld().items():
    cell.set_edgecolor("#cccccc")
    if row == 0:
        cell.set_facecolor("#ececec")
        cell.set_text_props(weight="bold")
    elif row % 5 == 0:
        cell.set_facecolor("#f7f7f7")
    else:
        cell.set_facecolor("white")

ax.set_title("Horizon Experiment Results", fontsize=11, pad=16)
fig.tight_layout()
fig.savefig(TABLES / "summary_metrics_table.png", dpi=200, facecolor="white")
plt.show()

## RPS across forecast horizons

The larger ring marks h=20, the official M6 horizon.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for model, colour in COLOURS.items():
    block = summary[summary["model"] == model].sort_values("horizon_days")
    y = block["overall_mean_rps_excl_degenerate"]
    ax.plot(block["horizon_days"], y, marker="o", color=colour, label=SHORT[model])
    ax.plot([20], [y.iloc[-1]], marker="o", markersize=13,
            markerfacecolor="none", color=colour)

ax.axhline(NAIVE_RPS, color="grey", linestyle="--", linewidth=1)
ax.text(20, NAIVE_RPS + 0.0015, "naive 0.160", color="grey", ha="right", fontsize=9)
ax.set_xscale("log")
ax.set_xticks(HORIZONS)
ax.set_xticklabels([str(h) for h in HORIZONS])
ax.set_xlabel("Forecast horizon")
ax.set_ylabel("Mean RPS")
ax.set_title("RPS across forecast horizons")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "rps_vs_horizon.png", dpi=200)
plt.show()

## Confidence and entropy

Confidence is the largest of the five predicted probabilities. Entropy is the
Shannon entropy of the five-way forecast. Dashed lines mark the uniform forecast
[0.2, 0.2, 0.2, 0.2, 0.2]: confidence 0.2 and entropy ln 5 = 1.609.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
panels = [("mean_confidence", "Mean confidence", 0.2),
          ("mean_entropy", "Mean entropy", np.log(5))]
for ax, (column, title, reference) in zip(axes, panels):
    for model, colour in COLOURS.items():
        block = diagnostics[diagnostics["model"] == model].sort_values("horizon_days")
        ax.plot(block["horizon_days"], block[column], marker="o",
                color=colour, label=SHORT[model])
    ax.axhline(reference, color="grey", linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_xticks(HORIZONS)
    ax.set_xticklabels([str(h) for h in HORIZONS])
    ax.set_xlabel("Forecast horizon")
    ax.set_title(title)
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "confidence_entropy_vs_horizon.png", dpi=200)
plt.show()

## Per-asset confidence distribution

Outliers omitted. The dashed line is the uniform forecast, 0.2.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for ax, (model, colour) in zip(axes, COLOURS.items()):
    block = confidence[confidence["model"] == model]
    data = [block.loc[block["horizon_days"] == h, "confidence"] for h in HORIZONS]
    parts = ax.boxplot(data, tick_labels=[str(h) for h in HORIZONS], showfliers=False,
                       widths=0.55, patch_artist=True, medianprops={"color": "black"})
    for patch in parts["boxes"]:
        patch.set_facecolor(colour)
        patch.set_alpha(0.45)
    ax.axhline(0.2, color="grey", linestyle="--", linewidth=1)
    ax.set_xlabel("Forecast horizon")
    ax.set_title(SHORT[model], fontsize=10)
axes[0].set_ylabel("Confidence")
fig.tight_layout()
fig.savefig(FIGURES / "confidence_distributions.png", dpi=200)
plt.show()